# Multiple Linear Regression


The objective is to build a model that predicts sales based on the money spent on different marketing platforms such as television, radio, and newspapers.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn import metrics

plots_dir = Path("../../plots/etude2_RLmultiple")
plots_dir.mkdir(parents=True, exist_ok=True)


In [ ]:
df = pd.read_csv("advertising.csv")


0) Show the first 10 rows and the dimensions of df.

Hint: head


In [ ]:
display(df.head(10))
print(f"Dimensions: {df.shape}")


## Data Preprocessing


**1. Checking Missing Values**

Hint: isna()


In [ ]:
missing_values = df.isna().sum()
missing_values


**Question:** Does the database contain missing values?


Answer: The check below shows that the dataset contains no missing values.


**2. Checking for Duplicate Rows**

Hint: duplicated()


In [ ]:
duplicate_count = df.duplicated().sum()
print(f"Number of duplicated rows: {duplicate_count}")


**Question:** Are there duplicate rows in the database?


Answer: The check below shows whether duplicated rows are present.


**3. Checking for Outliers**

Hint: display boxplots.


In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
sns.boxplot(data=df, ax=ax)
ax.set_title("Boxplots for Advertising Variables")
fig.tight_layout()
fig.savefig(plots_dir / "01_boxplots.pdf", bbox_inches="tight")
plt.show()


Comment on the number of outliers.


Answer: The boxplots show a small number of outliers, mainly in Newspaper. They are not numerous enough to remove automatically without a domain reason.


## Exploratory Data Analysis


**4. Distribution of the Target Variable**


In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
sns.histplot(df["Sales"], kde=True, ax=ax)
ax.set_title("Distribution of Sales")
fig.tight_layout()
fig.savefig(plots_dir / "02_sales_distribution.pdf", bbox_inches="tight")
plt.show()


**Comment on the distribution of sales?**


Answer: Sales are spread over a moderate range and the distribution is not extremely skewed.


**5. How are sales related to the other variables?**


In [ ]:
g = sns.pairplot(df, x_vars=["TV", "Radio", "Newspaper"], y_vars="Sales", height=4, aspect=1, kind="scatter")
g.fig.suptitle("Sales Versus Advertising Channels", y=1.05)
g.fig.savefig(plots_dir / "03_sales_pairplot.pdf", bbox_inches="tight")
plt.show()


**Your comment?**


Answer: TV and Radio show a clear positive relationship with Sales. Newspaper has a much weaker visible relationship.


**6. Heatmap**


In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(df.corr(numeric_only=True), annot=True, cmap="coolwarm", center=0, ax=ax)
ax.set_title("Pearson Correlation Heatmap")
fig.tight_layout()
fig.savefig(plots_dir / "04_correlation_heatmap.pdf", bbox_inches="tight")
plt.show()


**Comment?**


Answer: Sales has the strongest correlation with TV, a positive correlation with Radio, and a weaker correlation with Newspaper.


## Model Building


Linear regression is a useful tool for predicting a quantitative response.

Prediction using:

    Simple linear regression

    Multiple linear regression


### Simple Linear Regression


Simple linear regression has only one x variable and one y variable. It is an approach for predicting a quantitative response using a single feature.

It establishes the relationship between two variables with a straight line. Linear regression tries to draw a line that is as close as possible to the data by finding the slope and intercept that define the line and minimize regression errors.

**Formula:** Y = β0 + β1X + e

    Y = Dependent variable / target variable
    β0 = Intercept of the regression line
    β1 = Slope coefficient
    X = Independent variable / predictive variable
    e = Error

**In our case:** Sales = β0 + β1*TV + e


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn import metrics


In [ ]:
x = df[['TV']]
y = df['Sales']

In [ ]:
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size = 0.3, random_state = 100)

In [ ]:
slr= LinearRegression()  
slr.fit(x_train, y_train)

In [ ]:
# Model coefficients
print("Intercept:", slr.intercept_)
print("Slope:", slr.coef_)


In [ ]:
print('Regression equation: Sales = {0}*TV + {1}'.format(round(slr.coef_[0], 3), round(slr.intercept_, 3)))


In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(x_train, y_train, label="Train data")
ax.plot(x_train, slr.predict(x_train), color="red", label="Regression line")
ax.set_xlabel("TV")
ax.set_ylabel("Sales")
ax.set_title("Simple Linear Regression: Sales vs TV")
ax.legend()
fig.tight_layout()
fig.savefig(plots_dir / "05_simple_regression_tv.pdf", bbox_inches="tight")
plt.show()


In [ ]:
# Predict results for train and test
y_pred_slr = slr.predict(x_test)
x_pred_slr = slr.predict(x_train)


In [ ]:
# Actual value and predicted value
slr_diff = pd.DataFrame({"Actual value": y_test, "Predicted value": y_pred_slr})
slr_diff


In [ ]:
# Display the score (coefficient of determination R²)
print("Model R² coefficient: {:.2f}".format(slr.score(x, y) * 100))


**Conclusion:** 81.10% of the data match the regression model.


In [ ]:
# 0 means the model is perfect, so the value should be as close to 0 as possible.
meanAbErr = metrics.mean_absolute_error(y_test, y_pred_slr)
meanSqErr = metrics.mean_squared_error(y_test, y_pred_slr)
rootMeanSqErr = np.sqrt(metrics.mean_squared_error(y_test, y_pred_slr))

print("Mean Absolute Error:", meanAbErr)
print("Mean Square Error:", meanSqErr)
print("Root Mean Square Error:", rootMeanSqErr)


### Multiple Linear Regression


Multiple linear regression (MLR) has one y variable and two or more x variables. It is an extension of simple linear regression because more than one predictive variable is needed to predict the response variable.

Multiple linear regression is one of the main regression algorithms that models the linear relationship between one continuous dependent variable and several independent variables.

Assumptions for multiple linear regression:
    1. A linear relationship must exist between the target and the predictive variables.
    2. The regression residuals should be normally distributed.
    3. MLR assumes that multicollinearity (correlation between independent variables) in the data is low or absent.

**Formula:** Y = β0 + β1X1 + β2X2 + β3X3 + ... + βnXn + e

    Y = Dependent variable / target variable
    β0 = Intercept
    β1, β2, ..βn = slope coefficients
    X1, X2, ..Xn = Independent variables / predictive variables
    e = Error

**For our case:** Sales = β0 + (β1 * TV) + (β2 * Radio) + (β3 * Newspaper) + e


**7. Define x and y**


In [ ]:
X = df[["TV", "Radio", "Newspaper"]]
y = df["Sales"]
display(X.head())
display(y.head())


**8. Split into train and test**


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=100)
print(X_train.shape, X_test.shape)


**9. Train the multiple linear regression model**


In [ ]:
mlr = LinearRegression()
mlr.fit(X_train, y_train)
mlr


**10. Display the intercept, the MLR model coefficients, and the corresponding feature names**


In [ ]:
coefficients = pd.DataFrame({"feature": X.columns, "coefficient": mlr.coef_})
print(f"Intercept: {mlr.intercept_:.4f}")
display(coefficients)


**11. Predict values for train and test**


In [ ]:
y_pred_train_mlr = mlr.predict(X_train)
y_pred_test_mlr = mlr.predict(X_test)


**12. Display the table of actual and predicted values for test**


In [ ]:
mlr_diff = pd.DataFrame({"Actual value": y_test, "Predicted value": y_pred_test_mlr})
mlr_diff.head(10)


**13. Display the R² coefficient for the database**


In [ ]:
print("R2 coefficient on all data: {:.2f}%".format(mlr.score(X, y) * 100))
print("R2 coefficient on test data: {:.2f}%".format(mlr.score(X_test, y_test) * 100))


**Comment?**


Answer: The multiple regression score is higher than the simple TV-only model because it uses the additional information from Radio and Newspaper.


**14. Display MAE, MSE, and RMSE errors**


In [ ]:
mae = metrics.mean_absolute_error(y_test, y_pred_test_mlr)
mse = metrics.mean_squared_error(y_test, y_pred_test_mlr)
rmse = np.sqrt(mse)
errors_mlr = pd.DataFrame({"metric": ["MAE", "MSE", "RMSE"], "value": [mae, mse, rmse]})
display(errors_mlr)

fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(y_test, y_pred_test_mlr, color="steelblue")
lims = [min(y_test.min(), y_pred_test_mlr.min()), max(y_test.max(), y_pred_test_mlr.max())]
ax.plot(lims, lims, color="red", linestyle="--")
ax.set_xlabel("Actual Sales")
ax.set_ylabel("Predicted Sales")
ax.set_title("Multiple Regression: Actual vs Predicted")
fig.tight_layout()
fig.savefig(plots_dir / "06_mlr_actual_vs_predicted.pdf", bbox_inches="tight")
plt.show()
